# Experiment 1: Multi-source behavioural credit risk modelling

**Question:** Does historical credit and repayment behaviour improve the detection of high-risk applicants beyond
application information?

**Data:** Home Credit Default Risk, shared 100,000-applicant sample (stratified). Shared split: train 70% /
val 15% / test 15%. All choices are made on train or val. Test is only used for the final comparisons.

| Section | What it does |
|---|---|
| 0. Setup | Loads the shared data, splits and evaluation code |
| 1. Gradient boosting | Application only vs all sources, then tuned on validation |
| 2. Logistic regression | Interpretable baseline, C tuned on validation |
| 3. Ablation | One source at a time, same settings |
| 4. Richer features | Adds behavioural features (POS/cash, card, instalments, bureau, previous) and repeats the ablation |
| 5. Feature selection | Removes constant and correlated features, then refits |
| 6. Final model | The model we think is best, with a summary of all models |

Outputs (score files and tables) are written to `exp1_outputs/`. Run all cells from top to bottom.

## 0. Setup
Shared data, shared split and the shared evaluation code (`eval_exp1.py`), used for every model below.

In [1]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

SHARED = Path("hongwei_outputs/outputs").resolve()   # shared cleaned data and evaluation code
CLEAN = SHARED / "clean_data"
HERE = Path("exp1_outputs").resolve()                   # score files and tables for this notebook
HERE.mkdir(exist_ok=True)

splits = pd.read_csv(CLEAN / "splits.csv")
print(splits["split"].value_counts().to_string())

spec = importlib.util.spec_from_file_location("eval_exp1", SHARED / "exp1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

# Gradient boosting settings chosen by tuning in Section 1 (all sources, validation PR-AUC)
PARAMS = dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0)
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols, data):
    return pd.get_dummies(data[cols], dummy_na=True).astype("float32")

split
train    69999
val      15001
test     15000


## 1. Gradient boosting: application only vs all sources, then tuned on validation

* **Features:** application table (120 features, gender and marital status excluded) and history aggregates
  from bureau, previous applications, instalments and credit cards (13 features).
* **Model:** HistGradientBoostingClassifier, default settings. Trained on train, scored on val and test.
* **Tuning:** 24 settings (learning rate, max leaf nodes, min samples per leaf, L2), chosen by validation PR-AUC.
  The number of trees is set by early stopping inside train.

In [2]:
EXCLUDE = ["SK_ID_CURR", "TARGET", "split", "CODE_GENDER", "NAME_FAMILY_STATUS"]

app = pd.read_parquet(CLEAN / "application.parquet").drop(columns=["TARGET"])
app = app.merge(splits[["SK_ID_CURR", "TARGET", "split"]], on="SK_ID_CURR", how="inner")
app_cols = [c for c in app.columns if c not in EXCLUDE]
print(f"{len(app):,} applicants, {len(app_cols)} application features")

100,000 applicants, 120 application features


In [3]:
# Bureau: loans at other lenders
b = pd.read_parquet(CLEAN / "bureau.parquet")
bureau = b.groupby("SK_ID_CURR").agg(
    n_bureau=("SK_ID_BUREAU", "size"),
    bureau_debt=("AMT_CREDIT_SUM_DEBT", "sum"),
    bureau_overdue=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    bureau_credit=("AMT_CREDIT_SUM", "sum"),
)
bureau["bureau_debt_ratio"] = bureau["bureau_debt"] / bureau["bureau_credit"].where(bureau["bureau_credit"] > 0)

# Previous Home Credit applications (duplicate submissions removed)
p = pd.read_parquet(CLEAN / "previous_application.parquet")
p = p[p["IS_DUPLICATE_SUBMISSION"] == 0].copy()
p["refused"] = (p["NAME_CONTRACT_STATUS"].astype(str) == "Refused").astype(int)
prev = p.groupby("SK_ID_CURR").agg(
    n_prev=("SK_ID_PREV", "size"),
    prev_refusal_rate=("refused", "mean"),
    prev_amt_mean=("AMT_APPLICATION", "mean"),
)

# Instalments: one row per instalment (split payments already merged in cleaning)
i = pd.read_parquet(CLEAN / "installments.parquet")
paid = i["DAYS_ENTRY_PAYMENT"].notna()
i["late"] = np.where(paid, (i["DAYS_ENTRY_PAYMENT"] - i["DAYS_INSTALMENT"]) > 0, np.nan)   # unpaid: not counted
i["under"] = (i["AMT_INSTALMENT"] - i["AMT_PAYMENT"].fillna(0)).clip(lower=0)
inst = i.groupby("SK_ID_CURR").agg(
    n_inst=("SK_ID_PREV", "size"),
    late_rate=("late", "mean"),
    underpay_mean=("under", "mean"),
)

# Credit cards: utilisation
c = pd.read_parquet(CLEAN / "credit_card.parquet")
c["util"] = c["AMT_BALANCE"] / c["AMT_CREDIT_LIMIT_ACTUAL"].where(c["AMT_CREDIT_LIMIT_ACTUAL"] > 0)
cc = c.groupby("SK_ID_CURR").agg(cc_util_mean=("util", "mean"), cc_months=("MONTHS_BALANCE", "nunique"))

hist = bureau.join([prev, inst, cc], how="outer").reset_index()
hist_cols = [col for col in hist.columns if col != "SK_ID_CURR"]
print(f"History features: {len(hist_cols)} columns for {len(hist):,} applicants with any history")

History features: 13 columns for 99,277 applicants with any history


In [4]:
feat = app.merge(hist, on="SK_ID_CURR", how="left")   # applicants with no history get NaN (model handles it)
assert feat["SK_ID_CURR"].is_unique

def fit_and_score(cols, name, seed=42):
    """Fit on train, score val + test, save SK_ID_CURR, score to <name>.csv."""
    X = pd.get_dummies(feat[cols], dummy_na=True).astype("float32")
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=seed)
    model.fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    path = HERE / f"{name}.csv"
    out.to_csv(path, index=False)
    print(f"{name}: {len(X.columns)} model inputs, {len(out):,} scored applicants -> {path.name}")
    return path

p_app = fit_and_score(app_cols, "exp1_app_only")
p_all = fit_and_score(app_cols + hist_cols, "exp1_all_sources")

exp1_app_only: 250 model inputs, 30,001 scored applicants -> exp1_app_only.csv


exp1_all_sources: 263 model inputs, 30,001 scored applicants -> exp1_all_sources.csv


In [5]:
spec = importlib.util.spec_from_file_location("eval_exp1", SHARED / "exp1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

table, comps, _ = ev.compare_prediction_files([p_app, p_all], ["Application only", "All sources"], n_boot=1000)
print("Test-set metrics (value with 95% CI); threshold chosen on validation")
display(ev.pretty_metrics(table))
print("Paired comparison vs the application-only baseline")
display(ev.pretty_comparisons(comps))

Test-set metrics (value with 95% CI); threshold chosen on validation


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Application only,0.132,0.738 (0.724-0.753),0.211 (0.191-0.230),0.208 (0.194-0.223),0.443 (0.415-0.469),0.283 (0.266-0.301),0.241 (0.219-0.263),0.298 (0.275-0.322)
All sources,0.153,0.748 (0.733-0.763),0.234 (0.214-0.256),0.231 (0.213-0.250),0.387 (0.359-0.415),0.289 (0.270-0.310),0.257 (0.235-0.279),0.319 (0.295-0.341)


Paired comparison vs the application-only baseline


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,All sources,Application only,roc_auc,+0.0095 (+0.0033 to +0.0151),0.002,significantly better
1,All sources,Application only,pr_auc,+0.0234 (+0.0116 to +0.0351),0,significantly better


### 1b. Tuning on validation

In [6]:
import itertools
from sklearn.metrics import average_precision_score, roc_auc_score

GRID = {
    "learning_rate": [0.03, 0.1],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [20, 100],
    "l2_regularization": [0.0, 1.0],
}
COMBOS = [dict(zip(GRID, v)) for v in itertools.product(*GRID.values())]   # 24 settings
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

def tune(cols, name):
    X = model_inputs(cols)
    tr, va = feat["split"] == "train", feat["split"] == "val"
    y_va = feat.loc[va, "TARGET"]
    rows = []
    for params in COMBOS:
        m = HistGradientBoostingClassifier(**params, **FIXED).fit(X[tr], feat.loc[tr, "TARGET"])
        s = m.predict_proba(X[va])[:, 1]
        rows.append({**params, "trees": m.n_iter_,
                     "val_roc_auc": roc_auc_score(y_va, s), "val_pr_auc": average_precision_score(y_va, s)})
    res = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
    print(f"{name}: top 5 of {len(COMBOS)} settings by validation PR-AUC")
    display(res.head(5).round(4))
    top = res.iloc[0]
    best = {"learning_rate": float(top["learning_rate"]), "max_leaf_nodes": int(top["max_leaf_nodes"]),
            "min_samples_leaf": int(top["min_samples_leaf"]), "l2_regularization": float(top["l2_regularization"])}
    print("Chosen:", best)
    return best, res

best_app, res_app = tune(app_cols, "Application only")
best_all, res_all = tune(app_cols + hist_cols, "All sources")

Application only: top 5 of 24 settings by validation PR-AUC


,learning_rate,max_leaf_nodes,min_samples_leaf,l2_regularization,trees,val_roc_auc,val_pr_auc
0,0.10,31,100,0.0,94,0.7317,0.2238
1,0.03,15,100,1.0,278,0.7367,0.2237
2,0.03,15,100,0.0,302,0.7364,0.2236
3,0.03,31,100,0.0,209,0.7369,0.2235
4,0.03,31,100,1.0,302,0.7389,0.2226


Chosen: {'learning_rate': 0.1, 'max_leaf_nodes': 31, 'min_samples_leaf': 100, 'l2_regularization': 0.0}


All sources: top 5 of 24 settings by validation PR-AUC


,learning_rate,max_leaf_nodes,min_samples_leaf,l2_regularization,trees,val_roc_auc,val_pr_auc
0,0.03,15,100,1.0,322,0.7531,0.2383
1,0.03,31,20,1.0,324,0.7554,0.2358
2,0.03,15,100,0.0,364,0.7536,0.2352
3,0.10,15,100,1.0,162,0.7501,0.2350
4,0.03,15,20,1.0,468,0.7528,0.2339


Chosen: {'learning_rate': 0.03, 'max_leaf_nodes': 15, 'min_samples_leaf': 100, 'l2_regularization': 1.0}


In [7]:
def fit_tuned(cols, params, name, seed=42):
    """Refit the chosen settings on train only, score val + test, save to <name>.csv."""
    X = model_inputs(cols)
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(**params, **FIXED).fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    path = HERE / f"{name}.csv"
    out.to_csv(path, index=False)
    print(f"{name}: trees used = {model.n_iter_}, saved {path.name}")
    return path

p_app_t = fit_tuned(app_cols, best_app, "exp1_app_only_tuned")
p_all_t = fit_tuned(app_cols + hist_cols, best_all, "exp1_all_sources_tuned")

exp1_app_only_tuned: trees used = 94, saved exp1_app_only_tuned.csv


exp1_all_sources_tuned: trees used = 322, saved exp1_all_sources_tuned.csv


In [8]:
print("Application only: default vs tuned")
t1, c1, _ = ev.compare_prediction_files([p_app, p_app_t], ["Application only (default)", "Application only (tuned)"], n_boot=1000)
display(ev.pretty_metrics(t1))
display(ev.pretty_comparisons(c1))

print("All sources: default vs tuned")
t2, c2, _ = ev.compare_prediction_files([p_all, p_all_t], ["All sources (default)", "All sources (tuned)"], n_boot=1000)
display(ev.pretty_metrics(t2))
display(ev.pretty_comparisons(c2))

print("Tuned models: application only vs all sources")
t3, c3, _ = ev.compare_prediction_files([p_app_t, p_all_t], ["Application only (tuned)", "All sources (tuned)"], n_boot=1000)
display(ev.pretty_metrics(t3))
display(ev.pretty_comparisons(c3))

Application only: default vs tuned


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Application only (default),0.132,0.738 (0.724-0.753),0.211 (0.191-0.230),0.208 (0.194-0.223),0.443 (0.415-0.469),0.283 (0.266-0.301),0.241 (0.219-0.263),0.298 (0.275-0.322)
Application only (tuned),0.146,0.736 (0.721-0.750),0.211 (0.193-0.232),0.216 (0.200-0.232),0.402 (0.374-0.429),0.281 (0.262-0.299),0.244 (0.223-0.265),0.302 (0.279-0.324)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,Application only (tuned),Application only (default),roc_auc,-0.0025 (-0.0068 to +0.0015),0.24,no significant difference
1,Application only (tuned),Application only (default),pr_auc,+0.0003 (-0.0073 to +0.0071),1,no significant difference


All sources: default vs tuned


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
All sources (default),0.153,0.748 (0.733-0.763),0.234 (0.214-0.256),0.231 (0.213-0.250),0.387 (0.359-0.415),0.289 (0.270-0.310),0.257 (0.235-0.279),0.319 (0.295-0.341)
All sources (tuned),0.144,0.753 (0.739-0.767),0.237 (0.216-0.259),0.224 (0.208-0.241),0.424 (0.397-0.452),0.293 (0.275-0.313),0.263 (0.241-0.284),0.325 (0.301-0.347)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,All sources (tuned),All sources (default),roc_auc,+0.0052 (+0.0025 to +0.0081),0.00031,significantly better
1,All sources (tuned),All sources (default),pr_auc,+0.0028 (-0.0039 to +0.0092),0.41,no significant difference


Tuned models: application only vs all sources


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Application only (tuned),0.146,0.736 (0.721-0.750),0.211 (0.193-0.232),0.216 (0.200-0.232),0.402 (0.374-0.429),0.281 (0.262-0.299),0.244 (0.223-0.265),0.302 (0.279-0.324)
All sources (tuned),0.144,0.753 (0.739-0.767),0.237 (0.216-0.259),0.224 (0.208-0.241),0.424 (0.397-0.452),0.293 (0.275-0.313),0.263 (0.241-0.284),0.325 (0.301-0.347)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,All sources (tuned),Application only (tuned),roc_auc,+0.0172 (+0.0107 to +0.0232),1.5e-07,significantly better
1,All sources (tuned),Application only (tuned),pr_auc,+0.0259 (+0.0140 to +0.0363),0,significantly better


## 2. Logistic regression baseline (C tuned on validation)

The interpretable baseline the proposal asks for. Median imputation and standard scaling are fitted on train
only, inside a pipeline. The regularisation strength C is chosen by validation PR-AUC.

In [9]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

C_GRID = [0.01, 0.1, 1.0, 10.0]

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

def make_lr(C):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C, max_iter=2000))

def tune_lr(cols, name):
    X = model_inputs(cols)
    tr, va = feat["split"] == "train", feat["split"] == "val"
    y_va = feat.loc[va, "TARGET"]
    rows = []
    for C in C_GRID:
        m = make_lr(C).fit(X[tr], feat.loc[tr, "TARGET"])
        s = m.predict_proba(X[va])[:, 1]
        rows.append({"C": C, "val_roc_auc": roc_auc_score(y_va, s), "val_pr_auc": average_precision_score(y_va, s)})
    res = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
    print(f"{name}: C tuned on validation PR-AUC")
    display(res.round(4))
    best = float(res.iloc[0]["C"])
    print("Chosen C:", best)
    return best

C_app = tune_lr(app_cols, "LR application only")
C_all = tune_lr(app_cols + hist_cols, "LR all sources")

LR application only: C tuned on validation PR-AUC


,C,val_roc_auc,val_pr_auc
0,1.00,0.7243,0.1971
1,10.00,0.7242,0.1970
2,0.10,0.7245,0.1969
3,0.01,0.7242,0.1962


Chosen C: 1.0


LR all sources: C tuned on validation PR-AUC


,C,val_roc_auc,val_pr_auc
0,0.10,0.7396,0.2162
1,10.00,0.7394,0.2160
2,0.01,0.7396,0.2158
3,1.00,0.7393,0.2158


Chosen C: 0.1


In [10]:
def fit_lr_and_save(cols, C, name):
    X = model_inputs(cols)
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = make_lr(C).fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    path = HERE / f"{name}.csv"
    out.to_csv(path, index=False)
    print(f"{name}: C={C}, {len(out):,} scored applicants -> {path.name}")
    return path

p_lr_app = fit_lr_and_save(app_cols, C_app, "exp1_lr_app_only")
p_lr_all = fit_lr_and_save(app_cols + hist_cols, C_all, "exp1_lr_all_sources")

exp1_lr_app_only: C=1.0, 30,001 scored applicants -> exp1_lr_app_only.csv


exp1_lr_all_sources: C=0.1, 30,001 scored applicants -> exp1_lr_all_sources.csv


In [11]:
spec = importlib.util.spec_from_file_location("eval_exp1", SHARED / "exp1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

print("Logistic regression: application only vs all sources")
t1, c1, _ = ev.compare_prediction_files([p_lr_app, p_lr_all], ["LR application only", "LR all sources"], n_boot=1000)
display(ev.pretty_metrics(t1))
display(ev.pretty_comparisons(c1))

print("All sources: logistic regression vs gradient boosting (tuned, from notebook 01)")
t2, c2, _ = ev.compare_prediction_files([p_lr_all, HERE / "exp1_all_sources_tuned.csv"],
                                        ["LR all sources", "GBM all sources (tuned)"], n_boot=1000)
display(ev.pretty_metrics(t2))
display(ev.pretty_comparisons(c2))

Logistic regression: application only vs all sources


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
LR application only,0.151,0.733 (0.717-0.747),0.205 (0.186-0.225),0.213 (0.196-0.231),0.365 (0.338-0.392),0.269 (0.250-0.288),0.239 (0.217-0.261),0.296 (0.274-0.320)
LR all sources,0.156,0.741 (0.726-0.754),0.211 (0.192-0.232),0.228 (0.211-0.245),0.373 (0.347-0.400),0.283 (0.264-0.301),0.247 (0.226-0.269),0.306 (0.283-0.328)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,LR all sources,LR application only,roc_auc,+0.0080 (+0.0032 to +0.0125),0.0013,significantly better
1,LR all sources,LR application only,pr_auc,+0.0061 (-0.0010 to +0.0136),0.1,no significant difference


All sources: logistic regression vs gradient boosting (tuned, from notebook 01)


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
LR all sources,0.156,0.741 (0.726-0.754),0.211 (0.192-0.232),0.228 (0.211-0.245),0.373 (0.347-0.400),0.283 (0.264-0.301),0.247 (0.226-0.269),0.306 (0.283-0.328)
GBM all sources (tuned),0.144,0.753 (0.739-0.767),0.237 (0.216-0.259),0.224 (0.208-0.241),0.424 (0.397-0.452),0.293 (0.275-0.313),0.263 (0.241-0.284),0.325 (0.301-0.347)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,GBM all sources (tuned),LR all sources,roc_auc,+0.0125 (+0.0054 to +0.0198),0.00088,significantly better
1,GBM all sources (tuned),LR all sources,pr_auc,+0.0260 (+0.0149 to +0.0372),0,significantly better


## 3. Ablation: one source at a time (same settings)

Adds one source group at a time to the application features, using the Section 1 gradient boosting settings for
every row. The difference between rows shows what each source adds, not what tuning adds.

In [12]:
feat = app.merge(hist, on="SK_ID_CURR", how="left")
assert feat["SK_ID_CURR"].is_unique

BUREAU = ["n_bureau", "bureau_debt", "bureau_overdue", "bureau_credit", "bureau_debt_ratio"]
PREVIOUS = ["n_prev", "prev_refusal_rate", "prev_amt_mean"]
REPAYMENT = ["n_inst", "late_rate", "underpay_mean", "cc_util_mean", "cc_months"]
assert set(BUREAU + PREVIOUS + REPAYMENT) == set(hist_cols)

In [13]:
from sklearn.ensemble import HistGradientBoostingClassifier

PARAMS = dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0)
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

In [14]:
GROUPS = {
    "Application": app_cols,
    "+ Bureau": app_cols + BUREAU,
    "+ Previous": app_cols + PREVIOUS,
    "+ Repayment": app_cols + REPAYMENT,
    "All sources": app_cols + hist_cols,
}
FILES = {}
for name, cols in GROUPS.items():
    X = model_inputs(cols)
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(**PARAMS, **FIXED).fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    path = HERE / f"exp1_abl_{name.replace(' ', '_').replace('+', 'plus').lower()}.csv"
    out.to_csv(path, index=False)
    FILES[name] = path
    print(f"{name:<12} inputs={X.shape[1]:>4}  trees={model.n_iter_:>4}  -> {path.name}")

Application  inputs= 250  trees= 278  -> exp1_abl_application.csv


+ Bureau     inputs= 255  trees= 319  -> exp1_abl_plus_bureau.csv


+ Previous   inputs= 253  trees= 381  -> exp1_abl_plus_previous.csv


+ Repayment  inputs= 255  trees= 337  -> exp1_abl_plus_repayment.csv


All sources  inputs= 263  trees= 322  -> exp1_abl_all_sources.csv


In [15]:
spec = importlib.util.spec_from_file_location("eval_exp1", SHARED / "exp1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

labels = list(GROUPS)
table, comps, _ = ev.compare_prediction_files([FILES[l] for l in labels], labels, n_boot=1000)
print("Test-set metrics (value with 95% CI); threshold chosen on validation")
display(ev.pretty_metrics(table))
print("Each source group vs the application-only baseline")
display(ev.pretty_comparisons(comps))

Test-set metrics (value with 95% CI); threshold chosen on validation


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Application,0.156,0.741 (0.726-0.754),0.215 (0.197-0.236),0.228 (0.211-0.246),0.377 (0.350-0.403),0.284 (0.265-0.303),0.246 (0.224-0.268),0.305 (0.281-0.328)
+ Bureau,0.166,0.743 (0.728-0.757),0.221 (0.201-0.242),0.235 (0.217-0.254),0.357 (0.331-0.382),0.283 (0.264-0.302),0.250 (0.231-0.273),0.310 (0.288-0.332)
+ Previous,0.153,0.743 (0.728-0.757),0.221 (0.201-0.243),0.225 (0.208-0.243),0.384 (0.357-0.411),0.283 (0.265-0.303),0.245 (0.224-0.266),0.304 (0.281-0.327)
+ Repayment,0.153,0.751 (0.737-0.765),0.232 (0.211-0.254),0.228 (0.212-0.246),0.391 (0.366-0.418),0.288 (0.270-0.308),0.252 (0.231-0.275),0.312 (0.290-0.336)
All sources,0.144,0.753 (0.739-0.767),0.237 (0.216-0.259),0.224 (0.208-0.241),0.424 (0.397-0.452),0.293 (0.275-0.313),0.263 (0.241-0.284),0.325 (0.301-0.347)


Each source group vs the application-only baseline


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,+ Bureau,Application,roc_auc,+0.0023 (-0.0006 to +0.0053),0.13,no significant difference
1,+ Bureau,Application,pr_auc,+0.0052 (-0.0010 to +0.0108),0.1,no significant difference
2,+ Previous,Application,roc_auc,+0.0024 (-0.0007 to +0.0052),0.13,no significant difference
3,+ Previous,Application,pr_auc,+0.0060 (-0.0002 to +0.0110),0.062,no significant difference
4,+ Repayment,Application,roc_auc,+0.0109 (+0.0064 to +0.0153),7.5e-07,significantly better
5,+ Repayment,Application,pr_auc,+0.0164 (+0.0076 to +0.0242),0,significantly better
6,All sources,Application,roc_auc,+0.0125 (+0.0074 to +0.0173),5e-06,significantly better
7,All sources,Application,pr_auc,+0.0215 (+0.0106 to +0.0309),0,significantly better


## 4. Richer behavioural features

Adds 15 features from the proposal that were missing from Section 1:

* **POS/cash:** worst days past due, months past due, remaining instalments
* **Credit card:** months below the minimum payment, months past due, minimum-payment ratio, balance trend
* **Instalments:** worst and average days late, underpayment frequency, total paid to total due
* **Bureau:** worst days overdue, active loan count
* **Previous applications:** approval rate, applications in the last year

The earlier features are kept, so each group is a superset of Section 3. The ablation is repeated with these
features, using the same settings.

In [16]:
feat_old = app.merge(hist, on="SK_ID_CURR", how="left")
BUREAU = ["n_bureau", "bureau_debt", "bureau_overdue", "bureau_credit", "bureau_debt_ratio"]
PREVIOUS = ["n_prev", "prev_refusal_rate", "prev_amt_mean"]
REPAYMENT = ["n_inst", "late_rate", "underpay_mean", "cc_util_mean", "cc_months"]
assert set(BUREAU + PREVIOUS + REPAYMENT) == set(hist_cols)

In [17]:
# POS / cash loans
q = pd.read_parquet(CLEAN / "pos_cash.parquet", columns=["SK_ID_CURR", "SK_DPD", "CNT_INSTALMENT_FUTURE"])
q["dpd_flag"] = (q["SK_DPD"] > 0).astype("int32")
pos = q.groupby("SK_ID_CURR").agg(
    pos_max_dpd=("SK_DPD", "max"),
    pos_months_dpd=("dpd_flag", "sum"),
    pos_remaining_inst=("CNT_INSTALMENT_FUTURE", "mean"),
)

# Credit card, month by month (one row per applicant-month, summed over contracts)
c = pd.read_parquet(CLEAN / "credit_card.parquet", columns=[
    "SK_ID_CURR", "MONTHS_BALANCE", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_TOTAL_CURRENT", "SK_DPD"])
c["below_min"] = ((c["AMT_INST_MIN_REGULARITY"] > 0) &
                  (c["AMT_PAYMENT_TOTAL_CURRENT"].fillna(0) < c["AMT_INST_MIN_REGULARITY"])).astype("int32")
c["dpd_flag"] = (c["SK_DPD"] > 0).astype("int32")
c["min_ratio"] = c["AMT_PAYMENT_TOTAL_CURRENT"] / c["AMT_INST_MIN_REGULARITY"].where(c["AMT_INST_MIN_REGULARITY"] > 0)
m = c.groupby(["SK_ID_CURR", "MONTHS_BALANCE"], as_index=False).agg(
    bal=("AMT_BALANCE", "sum"), lim=("AMT_CREDIT_LIMIT_ACTUAL", "sum"),
    below=("below_min", "max"), dpd=("dpd_flag", "max"), min_ratio=("min_ratio", "mean"))
m["util"] = m["bal"] / m["lim"].where(m["lim"] > 0)
recent = m[m["MONTHS_BALANCE"] >= -3].groupby("SK_ID_CURR")["util"].mean()
card = m.groupby("SK_ID_CURR").agg(
    cc_months_below_min=("below", "sum"),
    cc_months_dpd=("dpd", "sum"),
    cc_min_ratio_mean=("min_ratio", "mean"),
    cc_util_overall=("util", "mean"),
)
card["cc_balance_trend"] = recent.reindex(card.index) - card["cc_util_overall"]
card = card.drop(columns="cc_util_overall")

# Instalments
i = pd.read_parquet(CLEAN / "installments.parquet", columns=[
    "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"])
paid = i["DAYS_ENTRY_PAYMENT"].notna()
i["days_late"] = np.where(paid, (i["DAYS_ENTRY_PAYMENT"] - i["DAYS_INSTALMENT"]).clip(lower=0), np.nan)
i["under_flag"] = ((i["AMT_INSTALMENT"] - i["AMT_PAYMENT"].fillna(0)) > 1.0).astype("int32")
i["paid_amt"] = i["AMT_PAYMENT"].fillna(0)
inst_new = i.groupby("SK_ID_CURR").agg(
    inst_max_days_late=("days_late", "max"),
    inst_mean_days_late=("days_late", "mean"),
    inst_underpay_freq=("under_flag", "mean"),
    paid_sum=("paid_amt", "sum"),
    due_sum=("AMT_INSTALMENT", "sum"),
)
inst_new["inst_paid_to_due"] = inst_new["paid_sum"] / inst_new["due_sum"].where(inst_new["due_sum"] > 0)
inst_new = inst_new.drop(columns=["paid_sum", "due_sum"])

# Bureau
b = pd.read_parquet(CLEAN / "bureau.parquet", columns=["SK_ID_CURR", "CREDIT_DAY_OVERDUE", "CREDIT_ACTIVE"])
bureau_new = b.groupby("SK_ID_CURR").agg(
    bureau_max_days_overdue=("CREDIT_DAY_OVERDUE", "max"),
    bureau_active_count=("CREDIT_ACTIVE", lambda s: (s.astype(str) == "Active").sum()),
)

# Previous applications
p = pd.read_parquet(CLEAN / "previous_application.parquet", columns=[
    "SK_ID_CURR", "NAME_CONTRACT_STATUS", "DAYS_DECISION", "IS_DUPLICATE_SUBMISSION"])
p = p[p["IS_DUPLICATE_SUBMISSION"] == 0].copy()
p["approved"] = (p["NAME_CONTRACT_STATUS"].astype(str) == "Approved").astype("int32")
p["last_year"] = (p["DAYS_DECISION"] >= -365).astype("int32")
prev_new = p.groupby("SK_ID_CURR").agg(prev_approval_rate=("approved", "mean"), prev_apps_last_year=("last_year", "sum"))

new_hist = pos.join([card, inst_new, bureau_new, prev_new], how="outer").reset_index()
NEW_POS = ["pos_max_dpd", "pos_months_dpd", "pos_remaining_inst"]
NEW_CARD = ["cc_months_below_min", "cc_months_dpd", "cc_min_ratio_mean", "cc_balance_trend"]
NEW_INST = ["inst_max_days_late", "inst_mean_days_late", "inst_underpay_freq", "inst_paid_to_due"]
NEW_BUREAU = ["bureau_max_days_overdue", "bureau_active_count"]
NEW_PREV = ["prev_approval_rate", "prev_apps_last_year"]
assert set(NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_PREV) == set(new_hist.columns) - {"SK_ID_CURR"}
print(f"New features: {len(new_hist.columns) - 1} columns for {len(new_hist):,} applicants")
display(new_hist.describe().T[["count", "mean", "min", "max"]].round(3))

New features: 15 columns for 99,290 applicants


,count,mean,min,max
SK_ID_CURR,99290.0,277928.762,100011.00,456248.000
pos_max_dpd,94164.0,15.262,0.00,4231.000
pos_months_dpd,94164.0,0.885,0.00,120.000
pos_remaining_inst,94155.0,9.215,0.00,59.500
cc_months_below_min,28368.0,2.096,0.00,70.000
cc_months_dpd,28368.0,1.546,0.00,94.000
cc_min_ratio_mean,19351.0,34.067,0.00,471288.531
cc_balance_trend,21320.0,0.007,-0.97,2.767
inst_max_days_late,94873.0,18.639,0.00,2852.000
inst_mean_days_late,94873.0,1.282,0.00,2813.000


In [18]:
feat = feat_old.merge(new_hist, on="SK_ID_CURR", how="left")
assert feat["SK_ID_CURR"].is_unique

GROUPS = {
    "Application": app_cols,
    "+ Bureau": app_cols + BUREAU + NEW_BUREAU,
    "+ Previous": app_cols + PREVIOUS + NEW_PREV,
    "+ Repayment": app_cols + REPAYMENT + NEW_POS + NEW_CARD + NEW_INST,
    "All sources": app_cols + hist_cols + NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_PREV,
}

In [19]:
from sklearn.ensemble import HistGradientBoostingClassifier

PARAMS = dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0)
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

FILES = {}
for name, cols in GROUPS.items():
    X = model_inputs(cols)
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(**PARAMS, **FIXED).fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    path = HERE / f"exp1_v2_{name.replace(' ', '_').replace('+', 'plus').lower()}.csv"
    out.to_csv(path, index=False)
    FILES[name] = path
    print(f"{name:<12} inputs={X.shape[1]:>4}  trees={model.n_iter_:>4}  -> {path.name}")

Application  inputs= 250  trees= 278  -> exp1_v2_application.csv


+ Bureau     inputs= 257  trees= 335  -> exp1_v2_plus_bureau.csv


+ Previous   inputs= 255  trees= 357  -> exp1_v2_plus_previous.csv


+ Repayment  inputs= 266  trees= 324  -> exp1_v2_plus_repayment.csv


All sources  inputs= 278  trees= 415  -> exp1_v2_all_sources.csv


In [20]:
spec = importlib.util.spec_from_file_location("eval_exp1", SHARED / "exp1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

labels = list(GROUPS)
table, comps, _ = ev.compare_prediction_files([FILES[l] for l in labels], labels, n_boot=1000)
display(ev.pretty_metrics(table))
display(ev.pretty_comparisons(comps))

,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Application,0.156,0.741 (0.726-0.754),0.215 (0.197-0.236),0.228 (0.211-0.246),0.377 (0.350-0.403),0.284 (0.265-0.303),0.246 (0.224-0.268),0.305 (0.281-0.328)
+ Bureau,0.158,0.743 (0.729-0.757),0.224 (0.205-0.245),0.231 (0.214-0.249),0.377 (0.349-0.403),0.286 (0.266-0.305),0.255 (0.234-0.278),0.316 (0.294-0.339)
+ Previous,0.154,0.742 (0.727-0.756),0.222 (0.203-0.244),0.225 (0.208-0.243),0.384 (0.358-0.411),0.284 (0.265-0.303),0.247 (0.226-0.269),0.306 (0.282-0.329)
+ Repayment,0.154,0.755 (0.741-0.770),0.240 (0.219-0.264),0.237 (0.220-0.255),0.400 (0.373-0.425),0.298 (0.278-0.316),0.263 (0.242-0.285),0.325 (0.301-0.348)
All sources,0.142,0.758 (0.743-0.772),0.251 (0.230-0.274),0.228 (0.211-0.244),0.443 (0.414-0.471),0.301 (0.282-0.319),0.271 (0.249-0.294),0.336 (0.313-0.359)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,+ Bureau,Application,roc_auc,+0.0029 (-0.0001 to +0.0060),0.076,no significant difference
1,+ Bureau,Application,pr_auc,+0.0085 (+0.0013 to +0.0146),0.014,significantly better
2,+ Previous,Application,roc_auc,+0.0016 (-0.0018 to +0.0046),0.32,no significant difference
3,+ Previous,Application,pr_auc,+0.0070 (+0.0008 to +0.0122),0.022,significantly better
4,+ Repayment,Application,roc_auc,+0.0150 (+0.0101 to +0.0202),4.9e-09,significantly better
5,+ Repayment,Application,pr_auc,+0.0250 (+0.0148 to +0.0344),0,significantly better
6,All sources,Application,roc_auc,+0.0175 (+0.0120 to +0.0229),4.7e-09,significantly better
7,All sources,Application,pr_auc,+0.0359 (+0.0236 to +0.0462),0,significantly better


In [21]:
feat.drop(columns=["split", "TARGET"]).to_parquet(HERE / "exp1_features_v2.parquet", index=False)
print("saved exp1_features_v2.parquet", feat.shape)

saved exp1_features_v2.parquet (100000, 153)


## 5. Feature selection

Checks the 278 model inputs before deciding what to keep. Every choice uses train only.

1. Drop features that are almost always empty, or have a single value.
2. Find numeric pairs with Spearman |rho| above 0.9. For each pair, keep the feature with the higher train AUC.
3. Refit the all-sources model on the selected features with the same settings, and compare.

In [22]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

SHARED = Path("hongwei_outputs/outputs").resolve()
CLEAN = SHARED / "clean_data"
HERE = Path("exp1_outputs").resolve()
THR = 0.9          # correlation threshold
MISS_MAX = 0.95    # drop features missing for more than this share of applicants

pass  # feat comes from Section 4
splits = pd.read_csv(CLEAN / "splits.csv")[["SK_ID_CURR", "TARGET", "split"]]
pass  # already has split and TARGET
tr = feat["split"] == "train"
y_tr = feat.loc[tr, "TARGET"]

EXCLUDE = ["SK_ID_CURR", "TARGET", "split", "CODE_GENDER", "NAME_FAMILY_STATUS"]
candidates = [c for c in feat.columns if c not in EXCLUDE]
numeric = [c for c in candidates if pd.api.types.is_numeric_dtype(feat[c])]
categorical = [c for c in candidates if c not in numeric]
print(f"{len(candidates)} candidate features: {len(numeric)} numeric, {len(categorical)} categorical")

148 candidate features: 134 numeric, 14 categorical


In [23]:
miss = feat[candidates].isna().mean().sort_values(ascending=False)
print("Most missing features (share of all applicants):")
display(miss.head(10).round(3).to_frame("missing_share"))

drop_missing = miss[miss > MISS_MAX].index.tolist()
nunique = feat[candidates].nunique(dropna=True)
drop_constant = nunique[nunique <= 1].index.tolist()
print("Dropped for missingness:", drop_missing or "none")
print("Dropped as constant:", drop_constant or "none")

kept = [c for c in candidates if c not in drop_missing + drop_constant]
numeric = [c for c in numeric if c in kept]
categorical = [c for c in categorical if c in kept]

Most missing features (share of all applicants):


,missing_share
cc_min_ratio_mean,0.806
cc_balance_trend,0.787
cc_util_mean,0.719
cc_months_below_min,0.716
cc_months_dpd,0.716
cc_months,0.716
COMMONAREA_AVG,0.701
COMMONAREA_MODE,0.701
COMMONAREA_MEDI,0.701
NONLIVINGAPARTMENTS_MEDI,0.697


Dropped for missingness: none
Dropped as constant: ['FLAG_MOBIL', 'FLAG_DOCUMENT_12']


In [24]:
X_tr = feat.loc[tr, numeric]
corr = X_tr.corr(method="spearman").abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().sort_values(ascending=False)
pairs = pairs[pairs > THR]
print(f"{len(pairs)} numeric pairs with |rho| > {THR}")
display(pairs.head(25).round(3).to_frame("abs_spearman"))

75 numeric pairs with |rho| > 0.9


,,abs_spearman
FLAG_EMP_PHONE,DAYS_EMPLOYED_ANOM,1.000
YEARS_BUILD_AVG,YEARS_BUILD_MEDI,0.998
YEARS_BEGINEXPLUATATION_AVG,YEARS_BEGINEXPLUATATION_MEDI,0.997
OBS_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,0.997
LIVINGAPARTMENTS_AVG,LIVINGAPARTMENTS_MEDI,0.996
FLOORSMIN_AVG,FLOORSMIN_MEDI,0.996
COMMONAREA_AVG,COMMONAREA_MEDI,0.996
LANDAREA_AVG,LANDAREA_MEDI,0.996
LIVINGAREA_AVG,LIVINGAREA_MEDI,0.995
APARTMENTS_AVG,APARTMENTS_MEDI,0.995


In [25]:
def univariate_auc(col):
    x = X_tr[col]
    x = x.fillna(x.median())
    a = roc_auc_score(y_tr, x)
    return max(a, 1 - a)

auc = {c: univariate_auc(c) for c in numeric}

dropped, rows = set(), []
for (a, b), r in pairs.items():
    if a in dropped or b in dropped:
        continue
    weaker, stronger = (a, b) if auc[a] < auc[b] else (b, a)
    dropped.add(weaker)
    rows.append({"dropped": weaker, "kept": stronger, "abs_spearman": round(r, 3),
                 "AUC_dropped": round(auc[weaker], 3), "AUC_kept": round(auc[stronger], 3)})
choices = pd.DataFrame(rows)
display(choices)
selected = [c for c in kept if c not in dropped]
print(f"Dropped {len(dropped)} correlated features; {len(selected)} features selected in total")

,dropped,kept,abs_spearman,AUC_dropped,AUC_kept
0,DAYS_EMPLOYED_ANOM,FLAG_EMP_PHONE,1.000,0.536,0.536
1,YEARS_BUILD_MEDI,YEARS_BUILD_AVG,0.998,0.511,0.511
2,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BEGINEXPLUATATION_AVG,0.997,0.520,0.520
3,OBS_60_CNT_SOCIAL_CIRCLE,OBS_30_CNT_SOCIAL_CIRCLE,0.997,0.507,0.508
4,LIVINGAPARTMENTS_MEDI,LIVINGAPARTMENTS_AVG,0.996,0.508,0.508
5,FLOORSMIN_MEDI,FLOORSMIN_AVG,0.996,0.513,0.513
6,COMMONAREA_MEDI,COMMONAREA_AVG,0.996,0.505,0.505
7,LANDAREA_MEDI,LANDAREA_AVG,0.996,0.507,0.507
8,LIVINGAREA_MEDI,LIVINGAREA_AVG,0.995,0.526,0.527
9,APARTMENTS_AVG,APARTMENTS_MEDI,0.995,0.519,0.520


Dropped 40 correlated features; 106 features selected in total


In [26]:
(HERE / "selected_features.txt").write_text("\n".join(selected))
print(f"saved selected_features.txt with {len(selected)} features")

saved selected_features.txt with 106 features


In [27]:
PARAMS = dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0)
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

full_cols = kept
sel_cols = selected
FILES = {}
for name, cols in [("All sources (full)", full_cols), ("All sources (selected)", sel_cols)]:
    X = model_inputs(cols)
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(**PARAMS, **FIXED).fit(X[tr], y_tr)
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    path = HERE / f"exp1_sel_{'full' if 'full' in name else 'selected'}.csv"
    out.to_csv(path, index=False)
    FILES[name] = path
    print(f"{name:<24} columns={len(cols):>3} dummy inputs={X.shape[1]:>4} trees={model.n_iter_:>4}")

All sources (full)       columns=146 dummy inputs= 276 trees= 415


All sources (selected)   columns=106 dummy inputs= 236 trees= 531


In [28]:
spec = importlib.util.spec_from_file_location("eval_exp1", SHARED / "exp1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

labels = list(FILES)
table, comps, _ = ev.compare_prediction_files([FILES[l] for l in labels], labels, n_boot=1000)
display(ev.pretty_metrics(table))
display(ev.pretty_comparisons(comps))

,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
All sources (full),0.142,0.758 (0.743-0.772),0.251 (0.230-0.274),0.228 (0.211-0.244),0.443 (0.414-0.471),0.301 (0.282-0.319),0.271 (0.249-0.294),0.336 (0.313-0.359)
All sources (selected),0.168,0.758 (0.743-0.773),0.247 (0.225-0.270),0.252 (0.232-0.272),0.373 (0.346-0.401),0.301 (0.280-0.322),0.263 (0.242-0.286),0.326 (0.304-0.349)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,All sources (selected),All sources (full),roc_auc,+0.0002 (-0.0021 to +0.0024),0.83,no significant difference
1,All sources (selected),All sources (full),pr_auc,-0.0046 (-0.0085 to -0.0009),0.026,significantly worse


## 6. Final model: all sources with the richer features

**Selected model:** gradient boosting on all application and history features, including the 15 new behavioural
features from Section 4 (278 model inputs after one-hot encoding), with the settings tuned in Section 1.

**Why this model:** it has the highest test ROC-AUC and PR-AUC of the models we built (Sections 1–5), and the
PR-AUC gain over application only is significant. The 106-feature version in Section 5 is a simpler alternative
with a small PR-AUC loss.

**Caveats:**
* Gender and marital status are excluded. Age is included.
* The new features were not separately tuned. Section 1 settings were reused.
* The model was picked by comparing test results across sections. A stricter approach would pick on validation.
  The validation results in Sections 1 and 4 show the same ranking.

The score file for Experiment 3 is `exp1_outputs/exp1_best_scores.csv` (applicants in val and test, columns
`SK_ID_CURR, score`).

In [29]:
best_file = HERE / "exp1_v2_all_sources.csv"
best = pd.read_csv(best_file)
best.to_csv(HERE / "exp1_best_scores.csv", index=False)
print(f"Best model scores: {len(best):,} applicants, saved exp1_best_scores.csv")

summary = {
    "GBM application only (default)": HERE / "exp1_app_only.csv",
    "GBM all sources, tuned (Section 1)": HERE / "exp1_all_sources_tuned.csv",
    "Logistic regression, all sources (Section 2)": HERE / "exp1_lr_all_sources.csv",
    "BEST: GBM all sources + 15 new features (Section 4)": best_file,
}
t, c, _ = ev.compare_prediction_files(list(summary.values()), list(summary.keys()), n_boot=1000)
print("Summary of Experiment 1 models (test set, 15,000 applicants)")
display(ev.pretty_metrics(t))
print("Each model vs application only (default gradient boosting)")
display(ev.pretty_comparisons(c))

Best model scores: 30,001 applicants, saved exp1_best_scores.csv


Summary of Experiment 1 models (test set, 15,000 applicants)


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
GBM application only (default),0.132,0.738 (0.724-0.753),0.211 (0.191-0.230),0.208 (0.194-0.223),0.443 (0.415-0.469),0.283 (0.266-0.301),0.241 (0.219-0.263),0.298 (0.275-0.322)
"GBM all sources, tuned (Section 1)",0.144,0.753 (0.739-0.767),0.237 (0.216-0.259),0.224 (0.208-0.241),0.424 (0.397-0.452),0.293 (0.275-0.313),0.263 (0.241-0.284),0.325 (0.301-0.347)
"Logistic regression, all sources (Section 2)",0.156,0.741 (0.726-0.754),0.211 (0.192-0.232),0.228 (0.211-0.245),0.373 (0.347-0.400),0.283 (0.264-0.301),0.247 (0.226-0.269),0.306 (0.283-0.328)
BEST: GBM all sources + 15 new features (Section 4),0.142,0.758 (0.743-0.772),0.251 (0.230-0.274),0.228 (0.211-0.244),0.443 (0.414-0.471),0.301 (0.282-0.319),0.271 (0.249-0.294),0.336 (0.313-0.359)


Each model vs application only (default gradient boosting)


,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,"GBM all sources, tuned (Section 1)",GBM application only (default),roc_auc,+0.0147 (+0.0090 to +0.0204),1.2e-06,significantly better
1,"GBM all sources, tuned (Section 1)",GBM application only (default),pr_auc,+0.0262 (+0.0143 to +0.0363),0,significantly better
2,"Logistic regression, all sources (Section 2)",GBM application only (default),roc_auc,+0.0023 (-0.0060 to +0.0107),0.61,no significant difference
3,"Logistic regression, all sources (Section 2)",GBM application only (default),pr_auc,+0.0002 (-0.0137 to +0.0124),0.98,no significant difference
4,BEST: GBM all sources + 15 new features (Secti...,GBM application only (default),roc_auc,+0.0197 (+0.0135 to +0.0257),1.5e-09,significantly better
5,BEST: GBM all sources + 15 new features (Secti...,GBM application only (default),pr_auc,+0.0406 (+0.0274 to +0.0523),0,significantly better
